# 02 · Bootstrap — Conversão do `.xlsx` para o formato de landing

**Por que este notebook existe:** o Auto Loader (`cloudFiles`) não lê
`.xlsx` nativamente — os formatos suportados são `csv`, `json`, `parquet`,
`avro`, `orc`, `text` e `binaryFile`. Como a extração atual da Locaweb
vem em Excel, este notebook faz uma conversão pontual (roda com Pandas no
driver, não é um job distribuído) de `.xlsx` → `.parquet`, gravando no
subdiretório do Volume que o Auto Loader efetivamente monitora.

**Por que Parquet, e não CSV**, para esse "arquivo ponte": no profiling
exploratório encontramos 11 quebras de linha e 1.241 aspas no campo
`Descrição resumida`, que quebram o parser CSV padrão do Spark se as
opções de `quote`/`escape`/`multiLine` não forem configuradas com
cuidado (chegamos a ler 122.554 linhas em vez de 122.543 até corrigir
isso). Parquet é *schema-safe* e elimina essa classe inteira de erro.

**Estrutura de pastas dentro do Volume:**
```
/Volumes/antecipeai/landing/raw/
incoming_xlsx/     <- onde vocês fazem upload manual do .xlsx (nunca lido pelo Auto Loader)
incidentes/        <- onde ESTE notebook grava o .parquet convertido (Auto Loader monitora aqui)
```

**Quando a Locaweb mandar extrações incrementais futuras:** se vierem em
CSV/Parquet, podem ser jogadas direto em `incidentes/`, pulando este
notebook — ele existe só por causa do formato Excel do arquivo atual.

**Pré-requisito:** ter subido `LW-DATASET.xlsx` para
`/Volumes/antecipeai/landing/raw/incoming_xlsx/` via Catalog Explorer
("Upload to this volume") antes de rodar este notebook.

In [0]:
%run ./00_config

## Garantir dependências (openpyxl para ler `.xlsx` via pandas)

In [0]:
%pip install -q openpyxl
dbutils.library.restartPython()

In [0]:
%run ./00_config

## Ler o `.xlsx` da área de upload manual

In [0]:
dbutils.widgets.text("source_filename", "LW-DATASET.xlsx", "Nome do arquivo .xlsx enviado")
source_filename = dbutils.widgets.get("source_filename")

# As pastas incoming_xlsx/ e incidentes/ já foram criadas no notebook 01
# (setup) — de propósito, para existirem antes de você precisar fazer o
# upload manual do .xlsx pelo Catalog Explorer.
incoming_dir = volume_path("incoming_xlsx")
incidentes_dir = volume_path("incidentes")

source_path = f"{incoming_dir}/{source_filename}"
print("Lendo arquivo de origem:", source_path)

import pandas as pd

pdf = pd.read_excel(source_path, sheet_name="Dataset Geral")
print("Shape lido:", pdf.shape)
print(pdf.dtypes)

In [0]:
# Resultado esperado (validado previamente fora do Databricks, com o dataset
# real da Locaweb): shape = (122543, 19), batendo exatamente com o dicionário
# de dados v2 (19 colunas: Número, Prioridade, Produto, Categoria,
# Subcategoria, Grupo designado, Item de configuração, Aberto, Resolvido,
# Encerrado, Duração, Código de fechamento, Descrição resumida, Solução,
# Aberto por, Incidente Pai, Status, Entrou para KPI?, KPI Violado?).
# Se o shape aqui vier diferente, o arquivo enviado não é o esperado —
# conferir antes de continuar.

## Converter tudo para string (schema-on-read mínimo)

A Bronze deve receber os dados "na íntegra", sem qualquer tratamento de
tipo — isso inclui não deixar o Parquet "adivinhar" tipos numéricos ou de
data. Forçamos todas as colunas para string aqui, na ponte, para que o
Auto Loader herde exatamente essa mesma fidelidade ao ler o Parquet na
Bronze (notebook `03`).

In [0]:
pdf_str = pdf.astype(str)
# pandas converte NaN para a string literal "nan" no astype(str) — desfazemos
# isso para não confundir "nulo de verdade" com o texto "nan" na Bronze.
pdf_str = pdf_str.where(pdf.notna(), None)

## Gravar como Parquet no diretório monitorado pelo Auto Loader

Forçamos um schema Parquet explícito com **todas as colunas como
string** (via `pyarrow.schema`), em vez de deixar o pandas/pyarrow
inferir o tipo. Isso evita um problema sutil de schema evolution no Auto
Loader: colunas com muitos nulos (ex.: `Resolvido`, presente em só 32,8%
das linhas) podem ser inferidas como tipo `null`/`double` num arquivo e
como `string` em outro, se cada extração futura for convertida
separadamente — o que quebraria a leitura incremental.

In [0]:
import uuid
from datetime import datetime, timezone
import pyarrow as pa
import pyarrow.parquet as pq

output_filename = f"incidentes_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}.parquet"
output_path = f"{incidentes_dir}/{output_filename}"

forced_string_schema = pa.schema([(col, pa.string()) for col in pdf_str.columns])
arrow_table = pa.Table.from_pandas(pdf_str, schema=forced_string_schema, preserve_index=False)
pq.write_table(arrow_table, output_path.replace("dbfs:", ""))

print(f"Arquivo convertido gravado em: {output_path}")
print(f"Linhas gravadas: {len(pdf_str)}")

## Conferência

In [0]:
display(dbutils.fs.ls(incidentes_dir))